In [1]:
from Utils.hdfs import init
from pyspark.sql import SparkSession
from pyspark.sql import types as T
from pyspark.sql import functions as F

In [2]:
builder = (
    SparkSession.builder
    .appName("data_and_schema_validation")
    .master("yarn")
    # .config("spark.yarn.am.memory", "512m")
    # .config("spark.driver.memory", "512m")
    # .config("spark.executor.memory", "512m")
    # .config("spark.executor.cores", "1")
    # .config("spark.executor.instances", "1")
    .config("spark.dynamicAllocation.enabled", "true")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.0")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.catalogImplementation", "hive")
    .config("spark.hadoop.hive.metastore.uris", "thrift://hive-metastore:9083")
    .enableHiveSupport()
)
spark = builder.getOrCreate()

In [3]:
init(spark)

In [4]:
%%hdfs
ls /user/jovyan/lakehouse/bronze/flight

PERMISSIONS  REPL OWNER      GROUP                 SIZE MODIFIED             NAME
------------------------------------------------------------------------------------------------------------------------
drwxrwxr-x   0    jovyan     supergroup               0 2026-07-05 14:08:32.664000 _delta_log
-rw-r--r--   1    jovyan     supergroup        79346802 2026-07-05 14:08:18.214000 part-00000-a5fd3b9f-5238-418d-9747-51bf1e484cc1-c000.snappy.parquet
-rwxrwxr-x   1    jovyan     supergroup        79409001 2026-07-05 12:29:32.277000 part-00000-d5be1eec-c9e9-4e28-b23f-4fecae99388f-c000.snappy.parquet
-rwxrwxr-x   1    jovyan     supergroup        79409001 2026-07-05 13:26:02.997000 part-00000-e388b146-4016-4742-9ed8-b4607ada5a4b-c000.snappy.parquet
-rwxrwxr-x   1    jovyan     supergroup        79427307 2026-07-05 12:29:42.967000 part-00001-0b21788d-112a-4b7a-99f5-f2341d2ee72c-c000.snappy.parquet
-rwxrwxr-x   1    jovyan     supergroup        79427307 2026-07-05 13:26:15.548000 part-00001-c9697

In [10]:
df_raw = spark.read.table('warehouse.flight_raw')

In [11]:
df_raw = df_raw.dropDuplicates()

In [12]:
df_stage = df_raw\
    .withColumn('departure_time', F.date_format(F.to_timestamp('departure_time', 'HH:mm:ss'), 'HH:mm:ss'))\
    .withColumn('departure_seconds', F.hour(F.to_timestamp('departure_time', 'HH:mm:ss')) * 3600 +\
                                     F.minute(F.to_timestamp('departure_time', 'HH:mm:ss')) * 60 +\
                                     F.second(F.to_timestamp('departure_time', 'HH:mm:ss')))\
    .withColumn('arrival_time', F.col('arrival_time').cast(T.TimestampType()))\
    .withColumn('flight_id', F.col('flight_id').cast('string'))\
    .withColumn('aircraft_id', F.col('aircraft_id').cast('string'))\
    .withColumn('itinerary_no', F.col('itinerary_no').cast('int'))\
    .withColumn('ticket_no', F.col('ticket_no').cast('string'))\
    .withColumn('flight_cost', F.col('flight_cost').cast(T.DecimalType()))\
    .withColumn('origin_airport', F.col('origin_airport').cast('string'))\
    .withColumn('destination_airport', F.col('destination_airport').cast('string'))\
    .withColumn('frequent_flier', F.col('frequent_flier').cast(T.BooleanType()))\
    .withColumn('travel_date', F.col('travel_date').cast('date'))\
    .withColumn('airplane_model', F.col('airplane_model').cast('string'))\
    .withColumn('frequent_flier_no', F.col('frequent_flier_no').cast('string'))\
    .withColumn('passenger_name', F.col('passenger_name').cast('string'))\
    .withColumn('passenger_country', F.col('passenger_country').cast('string'))\
    .withColumn('tail_no', F.col('tail_no').cast('string'))\
    .withColumn('distance', F.col('distance').cast('int'))\
    .withColumn('turbulance', F.col('turbulance').cast('int'))\
    .withColumn('temp_at_dept', F.col('temp_at_dept').cast('float'))\
    .withColumn('fuel_consumed_litre', F.col('fuel_consumed_litre').cast('float'))\
    .withColumn('taxi_duration_mins', F.col('taxi_duration_mins').cast('float'))\
    .withColumn('avg_flight_speed_kmps', F.col('avg_flight_speed_kmps').cast(T.DoubleType()))\
    .withColumn('engine_performance', F.col('engine_performance').cast('int'))\
    .withColumn('passenger_dob', F.col('passenger_dob').cast('date'))\
    .withColumn('passenger_flight_class', F.col('passenger_flight_class').cast(T.StringType()))

In [13]:
df_stage.select(F.count(df_stage.itinerary_no).alias('cnt'), F.count_distinct(df_stage.itinerary_no).alias('cnt_dist')).show()

+-------+--------+
|    cnt|cnt_dist|
+-------+--------+
|3000000| 2591255|
+-------+--------+



In [15]:
df_stage = df_stage.withColumn('uuid', F.expr('uuid()'))

In [16]:
%%sql
use warehouse;

""


In [18]:
df_stage.coalesce(2).write\
    .mode('append')\
    .format('delta')\
    .option('path', 'hdfs:///user/jovyan/lakehouse/silver/stage_flight')\
    .saveAsTable('flight')

In [ ]:
# In case it was not saved as a table we can use this

# %%sql

# create table if not exists warehouse.flight
# using delta
# location 'hdfs:///user/jovyan/lakehouse/silver/stage_flight';

In [19]:
spark.stop()